In [2]:
#library imports
import gcsfs
from google.cloud import storage
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import tensorflow as tf
import open3d as o3d
import numpy as np


/home/jacob/projects/waymo_research/3d_semseg/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jacob/projects/waymo_research/3d_semseg/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1779

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
#get google cloud token
import os
os.environ["CLOUDSDK_CONFIG"] = "/home/jacob/.config/gcloud"

import subprocess
token = subprocess.check_output(
    ["/usr/bin/gcloud", "auth", "print-access-token"]
).decode().strip()

In [4]:
from datetime import datetime, timezone, timedelta
fs = pafs.GcsFileSystem(access_token=token, credential_token_expiration=datetime.now(timezone.utc) + timedelta(hours=1))

#list files in each folder
folders = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/"))
lidar_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar/"))
seg_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_segmentation/"))
calib_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_calibration/"))

for f in lidar_files[:5]:
    print(f.path)

waymo_open_dataset_v_2_0_0/training/lidar/10017090168044687777_6380_000_6400_000.parquet
waymo_open_dataset_v_2_0_0/training/lidar/10023947602400723454_1120_000_1140_000.parquet
waymo_open_dataset_v_2_0_0/training/lidar/1005081002024129653_5313_150_5333_150.parquet
waymo_open_dataset_v_2_0_0/training/lidar/10061305430875486848_1080_000_1100_000.parquet
waymo_open_dataset_v_2_0_0/training/lidar/10072140764565668044_4060_000_4080_000.parquet


In [5]:
#sample lidar file loading to see contents
pf_lidar = pq.ParquetFile(lidar_files[0].path, filesystem=fs)
print("Number of row groups: ", pf_lidar.metadata.num_row_groups) 
print("Array of columns: ", pf_lidar.schema) 
# cols = ["key.frame_timestamp_micros", "[LiDARComponent].range_image_return1.tensor"]
rg = pf_lidar.read_row_group(0)
df = rg.to_pandas()
df.head(10)

Number of row groups:  4
Array of columns:  <pyarrow._parquet.ParquetSchema object at 0x78eedc575ec0>
required group field_id=-1 schema {
  optional binary field_id=-1 index (String);
  optional binary field_id=-1 key.segment_context_name (String);
  optional int64 field_id=-1 key.frame_timestamp_micros;
  optional int32 field_id=-1 key.laser_name (Int(bitWidth=8, isSigned=true));
  optional group field_id=-1 [LiDARComponent].range_image_return1.values (List) {
    repeated group field_id=-1 list {
      optional float field_id=-1 item;
    }
  }
  optional group field_id=-1 [LiDARComponent].range_image_return1.shape (List) {
    repeated group field_id=-1 list {
      optional int32 field_id=-1 item;
    }
  }
  optional group field_id=-1 [LiDARComponent].range_image_return2.values (List) {
    repeated group field_id=-1 list {
      optional float field_id=-1 item;
    }
  }
  optional group field_id=-1 [LiDARComponent].range_image_return2.shape (List) {
    repeated group field_id=-

,key.segment_context_name,key.frame_timestamp_micros,key.laser_name,[LiDARComponent].range_image_return1.values,[LiDARComponent].range_image_return1.shape,[LiDARComponent].range_image_return2.values,[LiDARComponent].range_image_return2.shape
index,,,,,,,
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,1,"[20.885931, 0.19042969, 1.493107, -1.0, 18.936...","[64, 2650, 4]","[-1.0, -1.0, -1.0, -1.0, 21.003038, 0.14941406...","[64, 2650, 4]"
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,2,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,3,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,4,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"
10017090168044687777_6380_000_6400_000;1550083467346370,10017090168044687777_6380_000_6400_000,1550083467346370,5,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"
10017090168044687777_6380_000_6400_000;1550083467446163,10017090168044687777_6380_000_6400_000,1550083467446163,1,"[18.391565, 0.031982422, 0.023421286, -1.0, 16...","[64, 2650, 4]","[-1.0, -1.0, -1.0, -1.0, 20.159872, 0.19238281...","[64, 2650, 4]"
10017090168044687777_6380_000_6400_000;1550083467446163,10017090168044687777_6380_000_6400_000,1550083467446163,2,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"
10017090168044687777_6380_000_6400_000;1550083467446163,10017090168044687777_6380_000_6400_000,1550083467446163,3,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"
10017090168044687777_6380_000_6400_000;1550083467446163,10017090168044687777_6380_000_6400_000,1550083467446163,4,"[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....","[200, 600, 4]"


In [ ]:
pf_calib = pq.ParquetFile(calib_files[0].path, filesystem=fs)
rgc = pf_calib.read_row_group(0)
df_rgc = rgc.to_pandas()
df_rgc.head(5)

,key.segment_context_name,key.laser_name,[LiDARCalibrationComponent].extrinsic.transform,[LiDARCalibrationComponent].beam_inclination.min,[LiDARCalibrationComponent].beam_inclination.max,[LiDARCalibrationComponent].beam_inclination.values
0,10017090168044687777_6380_000_6400_000,2,"[0.999762602122432, 0.0036877475559088514, 0.0...",-1.570796,0.523599,None
1,10017090168044687777_6380_000_6400_000,5,"[-0.9997324974348434, -0.013945152838121559, -...",-1.570796,0.523599,None
2,10017090168044687777_6380_000_6400_000,3,"[0.013387855309090483, -0.9997985726737589, 0....",-1.570796,0.523599,None
3,10017090168044687777_6380_000_6400_000,4,"[-0.026661403897150483, 0.9996313994371858, 0....",-1.570796,0.523599,None
4,10017090168044687777_6380_000_6400_000,1,"[-0.8477724631263563, -0.5303541574199493, -0....",-0.314504,0.039886,"[-0.30925746832197243, -0.29876530723528183, -..."


### Decoding the range images (spherical coordinates) into point clouds (cartesian coordinates)


In [ ]:
def sphere_to_cart(phi, rho, theta):
    horizontal_d = rho * np.cos(theta)
    X = horizontal_d * np.cos(phi)
    Y = horizontal_d * np.sin(phi)
    Z = rho * np.sin(theta)

    return X, Y, Z


laser_1_lidar = df.loc[df["key.laser_name"] == 1]
laser_1_lidar_t = laser_1_lidar.iloc[0]
#print(tuple(laser_1_lidar_t["[LiDARComponent].range_image_return1.shape"]))
#print(laser_1_lidar_t)

laser_1_lidar_t_grid = laser_1_lidar_t["[LiDARComponent].range_image_return1.values"].reshape(64, 2650, 4)
#print(laser_1_lidar_t_grid)


laser_1_calib = df_rgc.loc[df_rgc["key.laser_name"] == 1]
# extract the list of beam inclinations from the single row and convert to numpy array
theta_series = laser_1_calib["[LiDARCalibrationComponent].beam_inclination.values"].iloc[0]
theta_array = np.array(theta_series)
#print(theta_series)

phi_array = np.linspace(np.pi, np.pi * -1, num=2650)
#print(phi_array)

range_channel = laser_1_lidar_t_grid[:, :, 0]
ranges = range_channel[:, :]
range_mask = ranges > 0
print(range_mask)


(np.int32(64), np.int32(2650), np.int32(4))
[[ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 ...
 [False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]]


In [8]:
#apply to function

T_array, P_array = np.meshgrid(phi_array, theta_array)
# print(T_array)

X_unmasked, Y_unmasked, Z_unmasked = sphere_to_cart(P_array, range_channel, T_array)

X = X_unmasked[range_mask]
Y = Y_unmasked[range_mask]
Z = Z_unmasked[range_mask]

print("X: ", X)
print("Y: ", Y)
print("Z: ", Z)

X:  [-19.89509875 -18.03772642 -19.66062203 ...  -5.37526489  -5.63254863
  -5.77241344]
Y:  [ 6.35666265  5.76321551  6.28174523 ... -0.2069911  -0.2168986
 -0.22228453]
Z:  [ 2.55778886e-15  4.49146749e-02  9.79120532e-02 ... -2.38582871e+00
 -2.48404189e+00 -2.52937875e+00]


Using the extrensic transformation data to make the cartesian points global relative to the scene, instead of local

In [9]:
ex_transform_df= laser_1_calib["[LiDARCalibrationComponent].extrinsic.transform"].iloc[0]
ex_transform = np.array((ex_transform_df)).reshape(4, 4)
print(ex_transform)

points = np.column_stack((X, Y, Z))
homo_coords = np.column_stack((points, np.ones(len(X))))
print(points.shape)

homo_coords_T = np.transpose(homo_coords)
global_coords_T = np.dot(ex_transform, homo_coords_T)
global_coords_1 = np.transpose(global_coords_T)

#drop 4th column
global_coords = global_coords_1[:, :3]

[[-8.47772463e-01 -5.30354157e-01 -2.51365711e-03  1.43000000e+00]
 [ 5.30355440e-01 -8.47775368e-01  1.80144262e-04  0.00000000e+00]
 [-2.22655684e-03 -1.18041038e-03  9.99996825e-01  2.18400000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
(153830, 3)


Loading the labels

In [ ]:
pf_seg = pq.ParquetFile(seg_files[0].path, filesystem=fs)
seg = pf_seg.read_row_group(0)
df_seg = seg.to_pandas()
df_seg

,key.segment_context_name,key.frame_timestamp_micros,key.laser_name,[LiDARSegmentationLabelComponent].range_image_return1.values,[LiDARSegmentationLabelComponent].range_image_return1.shape,[LiDARSegmentationLabelComponent].range_image_return2.values,[LiDARSegmentationLabelComponent].range_image_return2.shape
index,,,,,,,
10017090168044687777_6380_000_6400_000;1550083469745187,10017090168044687777_6380_000_6400_000,1550083469745187,1,"[-1, 21, -1, 15, -1, 15, -1, 21, -1, 21, 0, 0,...","[64, 2650, 2]","[-1, 15, -1, 15, -1, 15, 0, 0, 0, 0, 0, 0, -1,...","[64, 2650, 2]"
10017090168044687777_6380_000_6400_000;1550083470245288,10017090168044687777_6380_000_6400_000,1550083470245288,1,"[-1, 17, -1, 17, -1, 17, -1, 17, -1, 21, -1, 2...","[64, 2650, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]"
10017090168044687777_6380_000_6400_000;1550083470745363,10017090168044687777_6380_000_6400_000,1550083470745363,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]"
10017090168044687777_6380_000_6400_000;1550083471245613,10017090168044687777_6380_000_6400_000,1550083471245613,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]"
10017090168044687777_6380_000_6400_000;1550083471746058,10017090168044687777_6380_000_6400_000,1550083471746058,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[64, 2650, 2]"


In [11]:
laser_1_seg = df_seg.loc[df_seg["key.laser_name"] == 1]
laser_1_seg_t = laser_1_seg.iloc[0]
#print(laser_1_seg_t)

laser_1_seg_t_grid = laser_1_seg_t["[LiDARSegmentationLabelComponent].range_image_return1.values"].reshape(64, 2650, 2)
print(laser_1_lidar_t_grid)

[[[ 2.08859310e+01  1.90429688e-01  1.49310696e+00 -1.00000000e+00]
  [ 1.89361095e+01  3.58886719e-02  7.61191770e-02 -1.00000000e+00]
  [ 2.06400089e+01  1.63085938e-01  9.30996120e-01 -1.00000000e+00]
  ...
  [ 1.95684834e+01  2.23632812e-01  1.17106430e-01 -1.00000000e+00]
  [ 1.57566700e+01  1.06811523e-02  8.19744989e-02 -1.00000000e+00]
  [ 2.09737606e+01  1.40625000e-01  8.90008867e-01 -1.00000000e+00]]

 [[ 1.73493176e+01  2.63671875e-02  5.26978932e-02 -1.00000000e+00]
  [ 1.74488583e+01  5.54199219e-02  1.11251108e-01 -1.00000000e+00]
  [ 1.73785934e+01  8.30078125e-02  6.44085333e-02 -1.00000000e+00]
  ...
  [ 1.84618282e+01  9.81445312e-02  1.75659642e-01 -1.00000000e+00]
  [ 2.00544758e+01  1.18652344e-01  8.84153545e-01 -1.00000000e+00]
  [ 2.04643478e+01  1.04492188e-01  4.45004433e-01 -1.00000000e+00]]

 [[ 1.99315147e+01  1.47460938e-01  1.03639185e+00 -1.00000000e+00]
  [ 2.07102718e+01  1.66992188e-01  4.68425713e-02 -1.00000000e+00]
  [ 1.79407043e+01  1.31835938e-

In [ ]:
seg_grid_masked = laser_1_seg_t_grid[range_mask]
seg_grid_masked_coords = seg_grid_masked[:, 1]

print(seg_grid_masked_coords)


(153830,)


Creating Functions for these methods to apply them to all LiDAR lasers

In [ ]:
#processing each lidar laser for a single timeframe

def laser_process(laser_num: int):

    #df == lidar folder (range image points)
    #df_rgc == calibration folder (angle of each lidar sensor)
    #df_seg == segmentation class labels

    #lidar folder processing
    laser_lidar = df.loc[df["key.laser_name"] == int(laser_num)]
    laser_lidar_t = laser_lidar.iloc[0]
    laser_shape = tuple(laser_lidar_t["[LiDARComponent].range_image_return1.shape"])
    laser_lidar_t_grid = laser_lidar_t["[LiDARComponent].range_image_return1.values"].reshape(laser_shape)

    #calibration folder processing
    laser_calib = df_rgc.loc[df_rgc["key.laser_name"] == int(laser_num)]

    if laser_calib["[LiDARCalibrationComponent].beam_inclination.values"].iloc[0] != None:
        theta_series = laser_calib["[LiDARCalibrationComponent].beam_inclination.values"].iloc[0]

    else:
        max_inclin = laser_calib["[LiDARCalibrationComponent].beam_inclination.max"].iloc[0]
        min_inclin = laser_calib["[LiDARCalibrationComponent].beam_inclination.min"].iloc[0]
        theta_series = np.linspace(max_inclin, min_inclin, num=laser_shape[0])
    
    #converting to cartesian from spherical range image
    theta_array = np.array(theta_series)
    phi_array = np.linspace(np.pi, np.pi * -1, num=laser_shape[1])

    #range channel / masking
    range_channel = laser_lidar_t_grid[:, :, 0]
    ranges = range_channel[:, :]
    range_mask = ranges > 0

    Theta_array, Phi_array = np.meshgrid(phi_array, theta_array)

    X_unmasked, Y_unmasked, Z_unmasked = sphere_to_cart(Phi_array, range_channel, Theta_array)

    X = X_unmasked[range_mask]
    Y = Y_unmasked[range_mask]
    Z = Z_unmasked[range_mask]
    
    #extrinsic transformation (to make it global relative to the scene instead of the sensor)
    ex_transform_df= laser_calib["[LiDARCalibrationComponent].extrinsic.transform"].iloc[0]
    ex_transform = np.array((ex_transform_df)).reshape(4, 4)

    points = np.column_stack((X, Y, Z))
    homo_coords = np.column_stack((points, np.ones(len(X))))

    homo_coords_T = np.transpose(homo_coords)
    global_coords_T = np.dot(ex_transform, homo_coords_T)
    global_coords_1 = np.transpose(global_coords_T)

    #drop 4th column to get the global X, Y, Z coords
    global_coords = global_coords_1[:, :3]

    #process segmentation labels 
    laser_seg = df_seg.loc[df_seg["key.laser_name"] == int(laser_num)]
    if not laser_seg.empty:
        laser_seg_t = laser_seg.iloc[0]
    else: 
        laser_seg_t = 
    seg_shape = tuple(laser_seg_t["[LiDARSegmentationLabelComponent].range_image_return1.shape"])
    laser_seg_t_grid = laser_seg_t["[LiDARSegmentationLabelComponent].range_image_return1.values"].reshape(seg_shape)
    
    #mask the seg grid to get the true segmentation labels of each X, Y, Z
    seg_grid_masked = laser_seg_t_grid[range_mask]
    seg_grid_masked_coords = seg_grid_masked[:, 1]

    return global_coords, seg_grid_masked_coords
